# 01 – Khảo sát cấu trúc dữ liệu (Bước 1)

**Mục tiêu:**
- Đọc header/metadata từ file IAGA-2002 và USGS CSV/GeoJSON
- Báo cáo: số dòng, số cột, kiểu dữ liệu, min/max/mean/std, khoảng thời gian
- Phân nhóm cột: thời gian / định danh / đo lường / chất lượng / metadata

**Nguyên tắc:** Chỉ đọc, KHÔNG sửa file trong `data/raw/`

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Thêm src vào path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import config as cfg
from parsers import (
    parse_iaga2002_file, parse_iaga2002_header,
    load_all_intermagnet, load_usgs_csv, load_usgs_geojson
)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_RAW:     {cfg.DATA_RAW}')

## 1. INTERMAGNET – Đọc header file đầu tiên

In [ ]:
# Đọc header của file .min đầu tiên tìm được cho mỗi trạm
# ĐÂY LÀ BƯỚC ĐỌC METADATA – không nạp toàn bộ dữ liệu

raw_intermagnet = cfg.DATA_RAW / 'intermagnet'

station_metas = {}
for station in cfg.STATIONS:
    files = sorted(raw_intermagnet.glob(f'intermagnet_{station}_*.min'))
    if not files:
        print(f'⚠  Không tìm thấy file cho trạm {station}')
        continue
    
    # Đọc header của file đầu tiên
    first_file = files[0]
    print(f'\n{'─'*60}')
    print(f'Trạm {station} | File: {first_file.name}')
    print(f'{'─'*60}')
    
    raw_text = first_file.read_text(encoding='ascii', errors='replace')
    header_lines = []
    for line in raw_text.splitlines():
        if line.strip().upper().startswith('DATE'):
            header_lines.append(line)
            break
        header_lines.append(line)
    
    # In header thô
    print('\n=== HEADER GỐC (30 dòng đầu) ===')
    for i, ln in enumerate(header_lines[:30]):
        print(f'{i+1:3d}: {ln}')
    
    # Parse metadata
    from parsers import parse_iaga2002_header
    meta = parse_iaga2002_header(header_lines)
    station_metas[station] = meta
    
    print('\n=== METADATA ĐÃ PARSE ===')
    for k, v in meta.items():
        print(f'  {k:30s}: {v}')

## 2. INTERMAGNET – Nạp dữ liệu và báo cáo thống kê

In [ ]:
# Nạp toàn bộ dữ liệu cho từng trạm
# Sentinel CHƯA được thay → giữ nguyên để đếm ở Bước 2

station_dfs   = {}  # Lưu DataFrame thô
station_meta2 = {}  # Lưu metadata

for station in cfg.STATIONS:
    try:
        df, meta = load_all_intermagnet(station)
        station_dfs[station]   = df
        station_meta2[station] = meta
        print(f'\n✅ {station}: nạp xong {len(df):,} dòng')
    except FileNotFoundError as e:
        print(f'⚠  {e}')

In [ ]:
# ── Báo cáo cấu trúc DataFrame ────────────────────────────────────────────
for station, df in station_dfs.items():
    meta = station_meta2.get(station, {})
    print(f'\n{"="*65}')
    print(f'  TRẠM {station} – Báo cáo cấu trúc')
    print(f'{"="*65}')
    
    print(f'\n📌 Tổng quan:')
    print(f'   Số dòng  : {len(df):,}')
    print(f'   Số cột   : {len(df.columns)}')
    print(f'   Cột      : {list(df.columns)}')
    
    t_min = df["time_utc"].min()
    t_max = df["time_utc"].max()
    print(f'   Từ       : {t_min}')
    print(f'   Đến      : {t_max}')
    print(f'   Span     : {t_max - t_min}')
    
    print(f'\n📌 Kiểu dữ liệu:')
    print(df.dtypes.to_string())
    
    print(f'\n📌 Thống kê mô tả (CÒN CHỨA SENTINEL):')
    print(df.describe(include='all').T.to_string())
    
    print(f'\n📌 5 dòng đầu:')
    print(df.head().to_string())

In [ ]:
# ── Phân nhóm cột ─────────────────────────────────────────────────────────
print('\n=== PHÂN NHÓM CỘT ===')
for station, df in station_dfs.items():
    print(f'\nTrạm {station}:')
    col_groups = {
        'Thời gian' : [c for c in df.columns if 'time' in c.lower() or 'doy' in c.lower()],
        'Định danh' : [c for c in df.columns if c in ('station',)],
        'Đo lường'  : [c for c in df.columns if c.endswith('_nt')],
        'Chất lượng': [c for c in df.columns if 'quality' in c.lower() or 'flag' in c.lower()],
        'Metadata'  : [c for c in df.columns if c not in 
                       sum([v for v in col_groups.values()], [])]
                      if 'col_groups' in dir() else [],
    }
    # Tính lại nhóm Metadata sau khi có 3 nhóm trên
    known = sum([col_groups[k] for k in list(col_groups.keys())[:-1]], [])
    col_groups['Metadata'] = [c for c in df.columns if c not in known]
    
    for group, cols in col_groups.items():
        print(f'   {group:12s}: {cols}')

## 3. USGS – Đọc CSV

In [ ]:
# Nạp USGS CSV
try:
    df_usgs_csv = load_usgs_csv()
    print(f'✅ USGS CSV: {len(df_usgs_csv):,} sự kiện địa chấn')
    print(f'   Từ: {df_usgs_csv["time_utc"].min()}')
    print(f'   Đến: {df_usgs_csv["time_utc"].max()}')
    print(f'\n📌 Kiểu dữ liệu:')
    print(df_usgs_csv.dtypes.to_string())
    print(f'\n📌 Thống kê mô tả:')
    print(df_usgs_csv.describe(include='all').T.to_string())
    print(f'\n📌 5 dòng đầu:')
    print(df_usgs_csv.head().to_string())
except FileNotFoundError as e:
    print(f'⚠  {e}')
    df_usgs_csv = None

## 4. USGS – Đọc GeoJSON

In [ ]:
# Nạp USGS GeoJSON
try:
    df_usgs_gj = load_usgs_geojson()
    print(f'✅ USGS GeoJSON: {len(df_usgs_gj):,} sự kiện địa chấn')
    print(f'   Từ: {df_usgs_gj["time_utc"].min()}')
    print(f'   Đến: {df_usgs_gj["time_utc"].max()}')
    print(f'\n📌 5 dòng đầu:')
    print(df_usgs_gj.head().to_string())
    
    # So sánh CSV vs GeoJSON
    if df_usgs_csv is not None:
        print(f'\n📌 So sánh CSV ({len(df_usgs_csv):,}) vs GeoJSON ({len(df_usgs_gj):,}):')
        print(f'   Chênh lệch: {len(df_usgs_csv) - len(df_usgs_gj)} sự kiện')
except FileNotFoundError as e:
    print(f'⚠  {e}')
    df_usgs_gj = None

## 5. Tóm tắt Bước 1

Chạy cell này sau khi đã chạy tất cả các cell trên để tổng hợp kết quả.

In [ ]:
print('=' * 65)
print('  TÓM TẮT BƯỚC 1 – KHẢO SÁT CẤU TRÚC')
print('=' * 65)

for station in cfg.STATIONS:
    if station not in station_dfs:
        continue
    df   = station_dfs[station]
    meta = station_meta2.get(station, {})
    meas = [c for c in df.columns if c.endswith('_nt')]
    
    print(f'\n[INTERMAGNET / {station}]')
    print(f'  Mã trạm     : {meta.get("station_code")}')
    print(f'  Toạ độ      : {meta.get("geodetic_lat")}°N, {meta.get("geodetic_lon")}°E')
    print(f'  Độ cao      : {meta.get("elevation")} m')
    print(f'  Thành phần  : {meta.get("reported_components")}')
    print(f'  Tần số      : {meta.get("sample_period_seconds")} giây')
    print(f'  Loại dữ liệu: {meta.get("data_type")}')
    print(f'  Sentinel    : {meta.get("sentinel_values")} (theo tài liệu INTERMAGNET)')
    print(f'  Số dòng     : {len(df):,}')
    print(f'  Khoảng TG   : {df["time_utc"].min()} → {df["time_utc"].max()}')
    print(f'  Cột đo lường: {meas}')

if df_usgs_csv is not None:
    print(f'\n[USGS CSV]')
    print(f'  Sự kiện     : {len(df_usgs_csv):,}')
    print(f'  Khoảng TG   : {df_usgs_csv["time_utc"].min()} → {df_usgs_csv["time_utc"].max()}')
    print(f'  Mag range   : {df_usgs_csv["magnitude"].min():.1f} – {df_usgs_csv["magnitude"].max():.1f}')
    print(f'  Cột         : {list(df_usgs_csv.columns)}')

print('\n' + '─'*65)
print('→ Xác nhận để tiếp tục Bước 2 (Kiểm tra chất lượng)')